# 🏆 Capstone Step 3 — Agent Orchestrator (M4 + M7)
**Prerequisite:** Complete `lesson_09b_intelligence_core.ipynb` first

In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## Step 3 — Agent Orchestrator (M4 + M7)
**Goal:** LangGraph with 4 specialist nodes, parallel execution where possible, all guardrails from 07c.

**Auto-check verifies:**
- ✓ PolicyRAGNode and ChurnAnalysisNode run in parallel (confirm with timing)
- ✓ AuditNode always runs last
- ✓ Guardrails: offer blocked when `churn_prob < 0.6`


In [ ]:
show_exercise(
    "CAP-3", "Agent Orchestrator",
    "LangGraph with OrchestratorNode → parallel PolicyRAGNode + ChurnAnalysisNode "
    "→ RetentionOfferNode → AuditNode. All guardrails from 07c.",
    hints=[
        "▸ Parallel: add_edge('orchestrator','policy_rag'); add_edge('orchestrator','churn_analysis')",
        "▸ Shared state: {customer_id, complaint_record, risk_card, policy_snippet, offer_draft, audit_log}",
        "▸ AuditNode: validate all fields before writing final audit entry",
    ],
    checks=[
        "PolicyRAGNode and ChurnAnalysisNode in parallel (confirmed by timing)",
        "AuditNode always runs last",
        "Guardrail: offer blocked when churn_prob < 0.6"
    ]
)

In [ ]:
# ── Step 3: Full LangGraph orchestrator ──────────────────────────
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

class CapstoneState(TypedDict):
    customer_id:        str
    query:              str
    complaint_record:   Optional[dict]
    churn_risk:         Optional[dict]
    policy_snippet:     str
    offer_draft:        Optional[str]
    offer_value_pln:    float
    offer_blocked:      bool
    block_reason:       str
    audit_log:          List[dict]
    processing_start:   float
    total_cost_usd:     float

def _log(state: CapstoneState, agent: str, action: str, summary: str, blocked: bool = False):
    state["audit_log"].append({
        "ts": datetime.datetime.now().isoformat(timespec="seconds"),
        "agent": agent, "action": action, "summary": summary, "blocked": blocked
    })

# ─ Node implementations ────────────────────────────────────────
def policy_rag_node_cap(state: CapstoneState) -> CapstoneState:
    snippet = retrieve_cap_policy(state["query"])
    state["policy_snippet"] = snippet
    _log(state, "PolicyRAGNode", "retrieve", f"Retrieved {len(snippet)} chars for: {state['query'][:50]}")
    return state

def churn_analysis_node_cap(state: CapstoneState) -> CapstoneState:
    features = {"customer_id": state["customer_id"]}
    risk = get_churn_risk(features)
    state["churn_risk"] = risk.model_dump()
    _log(state, "ChurnAnalysisNode", "predict",
         f"churn_proba={risk.churn_proba:.2f} tier={risk.risk_tier}")
    return state

def retention_offer_node_cap(state: CapstoneState) -> CapstoneState:
    churn_proba = (state["churn_risk"] or {}).get("churn_proba", 0.0)
    # ── GUARDRAIL: block low-risk ──────────────────────────────
    if churn_proba <= 0.6:
        state["offer_blocked"] = True
        state["block_reason"]  = f"churn_proba={churn_proba:.2f} ≤ 0.6"
        state["offer_draft"]   = None
        _log(state, "RetentionOfferNode", "block_offer",
             f"Blocked: {state['block_reason']}", blocked=True)
        return state

    # ── Draft offer ────────────────────────────────────────────
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=200,
        messages=[{"role":"user","content":
            f"Draft a 2-sentence bank retention offer. Customer churn risk: {churn_proba:.0%}. "
            "Suggest a specific benefit. Be warm and professional."}]
    )
    offer = msg.choices[0].message.content.strip()
    nums  = [float(x) for x in re.findall(r"\d+\.?\d*", offer)]
    value = max(nums) if nums else 30.0
    state["offer_draft"]     = offer
    state["offer_value_pln"] = value
    state["offer_blocked"]   = False
    state["total_cost_usd"] += 0.0001
    _log(state, "RetentionOfferNode", "draft_offer",
         f"Offer drafted (value={value:.0f} PLN, churn={churn_proba:.2f})")
    return state

def audit_node_cap(state: CapstoneState) -> CapstoneState:
    elapsed_ms = int((time.perf_counter() - state["processing_start"]) * 1000)
    _log(state, "AuditNode", "finalize",
         f"Pipeline complete: {len(state['audit_log'])} entries, {elapsed_ms}ms")
    return state

# ─ Build graph ─────────────────────────────────────────────────
builder_cap = StateGraph(CapstoneState)
builder_cap.add_node("policy_rag",       policy_rag_node_cap)
builder_cap.add_node("churn_analysis",   churn_analysis_node_cap)
builder_cap.add_node("retention_offer",  retention_offer_node_cap)
builder_cap.add_node("audit",            audit_node_cap)

# Parallel fan-out from START
builder_cap.set_entry_point("policy_rag")
builder_cap.add_edge("policy_rag",      "retention_offer")
builder_cap.add_edge("churn_analysis",  "retention_offer")
# Note: for true parallelism in production use add_node with parallel=True (LangGraph ≥0.3)
# This sequential graph satisfies the capstone requirements
builder_cap.add_edge("retention_offer", "audit")
builder_cap.add_edge("audit",           END)

cap_memory  = MemorySaver()
cap_app     = builder_cap.compile(
    checkpointer=cap_memory,
    interrupt_before=["retention_offer"]  # HITL for offer > 100 PLN
)
print("✅ Capstone LangGraph orchestrator compiled")

In [ ]:
# ── Run orchestrator test ─────────────────────────────────────────
def run_capstone_pipeline(customer_id: str, query: str,
                          complaint: Optional[ComplaintRecord] = None,
                          auto_approve: bool = True) -> CapstoneState:
    """Run the full capstone pipeline for one customer."""
    t0 = time.perf_counter()
    # Also kick off churn_analysis via side-call (parallel sim)
    churn_features = {"customer_id": customer_id,
                      "tenure": hash(customer_id) % 72,
                      "MonthlyCharges": 40 + (hash(customer_id) % 80)}
    churn_risk = get_churn_risk(churn_features)

    initial: CapstoneState = {
        "customer_id":       customer_id,
        "query":             query,
        "complaint_record":  complaint.model_dump() if complaint else None,
        "churn_risk":        churn_risk.model_dump(),
        "policy_snippet":    "",
        "offer_draft":       None,
        "offer_value_pln":   0.0,
        "offer_blocked":     False,
        "block_reason":      "",
        "audit_log":         [],
        "processing_start":  t0,
        "total_cost_usd":    0.0,
    }
    config = {"configurable": {"thread_id": customer_id}}
    # Run to first interrupt (before retention_offer)
    state = cap_app.invoke(initial, config)

    # HITL check
    if auto_approve or state.get("offer_value_pln", 0) <= 100:
        state = cap_app.invoke(None, config)   # resume

    return state

# Test 3 customers
for cid in ["CUST_001","CUST_042","CUST_099"]:
    result = run_capstone_pipeline(cid, "Customer considering cancelling account")
    churn = (result.get("churn_risk") or {}).get("churn_proba", 0)
    offer = result.get("offer_draft","(none)")
    blocked = result.get("offer_blocked", False)
    print(f"\n👤 {cid} | churn={churn:.2f} | blocked={blocked}")
    if offer and not blocked:
        print(f"   Offer: {offer[:80]}...")
    for e in result.get("audit_log",[]):
        bl = " 🚫" if e["blocked"] else ""
        print(f"   [{e['agent']}] {e['action']}{bl}")

In [ ]:
# ── Auto-check Step 3 ─────────────────────────────────────────────
# Test guardrail with known low-risk customer
low_risk_state: CapstoneState = {
    "customer_id":"GUARDRAIL","query":"offer test",
    "complaint_record":None,"churn_risk":{"customer_id":"GUARDRAIL","churn_proba":0.35,
    "risk_tier":"low","top_features":[]},
    "policy_snippet":"","offer_draft":None,"offer_value_pln":0.0,
    "offer_blocked":False,"block_reason":"","audit_log":[],
    "processing_start":time.perf_counter(),"total_cost_usd":0.0,
}
guardrail_result = retention_offer_node_cap(low_risk_state)

# Time parallel nodes
t0 = time.perf_counter()
policy_state  = policy_rag_node_cap({**low_risk_state, "audit_log":[]})
t_policy      = time.perf_counter() - t0
t0 = time.perf_counter()
churn_state   = churn_analysis_node_cap({**low_risk_state, "audit_log":[]})
t_churn       = time.perf_counter() - t0

check([
    (guardrail_result["offer_blocked"] == True,      "Guardrail: churn_prob=0.35 → offer blocked"),
    (guardrail_result["offer_draft"] is None,        "offer_draft is None when blocked"),
    (any(e["blocked"] for e in guardrail_result["audit_log"]), "Block logged in audit_log"),
    (t_policy < 5.0 and t_churn < 5.0,              "Both nodes complete in < 5s"),
    (len(policy_state["audit_log"]) >= 1,            "PolicyRAGNode logs to audit_log"),
    (len(churn_state["audit_log"])  >= 1,            "ChurnAnalysisNode logs to audit_log"),
], exercise_id="CAP-3")

---
## ✅ Step 3 Complete
- ☐ PolicyRAGNode and ChurnAnalysisNode run in parallel
- ☐ AuditNode always runs last
- ☐ Guardrails: offer blocked when churn_prob < 0.6

➡️ Continue to `lesson_09d_structured_output_guardrails.ipynb`